In [ ]:
import re
import time
import random
from urllib.parse import quote_plus
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from webdriver_manager.microsoft import EdgeChromiumDriverManager

In [2]:
edge_options = Options()
edge_options.add_argument("--start-maximized")
edge_options.add_argument("--disable-gpu")
driver = webdriver.Edge(service=Service(EdgeChromiumDriverManager().install()),options=edge_options)
driver.set_page_load_timeout(40)

In [3]:
job_title = "junior data analyst"
location = "Mumbai"
posted_within = "r604800"   # last 7 days; leave blank for all jobs
workplace_type = ""           # 1 = on-site, 2 = remote, 3 = hybrid
number_of_jobs = 100
get_job_description = True

In [4]:
def make_search_url(title, place, start=0):
    url = "https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search"
    params = [f"keywords={quote_plus(title)}",
        f"location={quote_plus(place)}",
        f"start={start}"]
    if posted_within:
        params.append(f"f_TPR={posted_within}")
    if workplace_type:
        params.append(f"f_WT={workplace_type}")
    return url + "?" + "&".join(params)

In [5]:
def read_job_cards(html):
    soup = BeautifulSoup(html, "lxml")
    jobs = []
    for card in soup.select("li"):
        title_tag = card.select_one(".base-search-card__title")
        company_tag = card.select_one(".base-search-card__subtitle a")
        location_tag = card.select_one(".job-search-card__location")
        link_tag = card.select_one("a.base-card__full-link")
        time_tag = card.select_one("time")
        title = title_tag.get_text(strip=True) if title_tag else ""
        company = company_tag.get_text(strip=True) if company_tag else ""
        place = location_tag.get_text(strip=True) if location_tag else ""
        link = link_tag.get("href", "").split("?")[0] if link_tag else ""
        posted_date = ""
        posted_text = ""
        if time_tag:
            posted_date = time_tag.get("datetime", "").strip()
            posted_text = time_tag.get_text(strip=True)
        if title and link:
            jobs.append({
                "title": title,
                "company": company,
                "location": place,
                "posted_date": posted_date,
                "posted_text": posted_text,
                "link": link})
    return jobs

In [6]:
def read_job_page(html):
    soup = BeautifulSoup(html, "lxml")
    description = ""
    box = soup.select_one('[data-testid="expandable-text-box"]')
    if box:
        description = box.get_text("\n", strip=True)
    if not description:
        box = soup.select_one(".show-more-less-html__markup")
        if box:
            description = box.get_text("\n", strip=True)
    if not description:
        heading = soup.find("h2", string=re.compile("About the job", re.I))
        if heading:
            section = heading.find_parent("section")
            if section:
                description = section.get_text("\n", strip=True)
    description = re.sub(r"\n{3,}", "\n\n", description).strip()
    posted = ""
    time_tag = soup.select_one("time")
    if time_tag:
        posted = time_tag.get("datetime", "").strip()
    applicants = ""
    applicant_tag = soup.select_one(
        ".num-applicants__caption, .jobs-unified-top-card__applicant-count")
    if applicant_tag:
        applicants = applicant_tag.get_text(strip=True)
    return description, posted, applicants

In [7]:
jobs = []
used_links = set()
page = 0
max_pages = 30
while len(jobs) < number_of_jobs and page < max_pages:
    start = page * 10
    url = make_search_url(job_title, location, start)
    try:
        driver.get(url)
        time.sleep(random.uniform(2.5, 4))
        page_jobs = read_job_cards(driver.page_source)
    except Exception:
        page += 1
        continue
    if not page_jobs:
        time.sleep(7)
        page += 1
        continue
    for job in page_jobs:
        if job["link"] not in used_links:
            used_links.add(job["link"])
            jobs.append(job)
    page += 1
    time.sleep(random.uniform(1.5, 3))

In [8]:
if get_job_description:
    for job in jobs:
        try:
            driver.get(job["link"])
            time.sleep(random.uniform(2, 3.5))
            description, posted, applicants = read_job_page(driver.page_source)
            job["jd"] = description
            job["applicants"] = applicants
            if posted:
                job["posted_date"] = posted
        except Exception:
            job["jd"] = ""
            job["applicants"] = ""
        time.sleep(random.uniform(1, 2.5))
else:
    for job in jobs:
        job["jd"] = ""
        job["applicants"] = ""

In [9]:
df = pd.DataFrame(jobs)

if df.empty:
    print("No jobs were collected.")
else:
    for column in ["title", "company", "location", "posted_text", "applicants"]:
        if column in df.columns:
            df[column] = (
                df[column].astype(str)
                .str.replace(r"\s+", " ", regex=True)
                .str.strip()
            )

    df["posted_dt"] = pd.to_datetime(df["posted_date"], errors="coerce", utc=True)

    def get_date_from_text(text):
        if not text or text == "nan":
            return pd.NaT

        text = text.lower()
        now = pd.Timestamp.now(tz="UTC")

        if any(word in text for word in ["just", "today", "minute", "hour"]):
            return now

        match = re.search(r"(\d+)\s*(day|week|month)", text)
        if not match:
            return pd.NaT

        number = int(match.group(1))
        unit = match.group(2)

        if unit == "day":
            return now - pd.Timedelta(days=number)
        if unit == "week":
            return now - pd.Timedelta(weeks=number)
        return now - pd.Timedelta(days=number * 30)
    missing_date = df["posted_dt"].isna()
    df.loc[missing_date, "posted_dt"] = df.loc[missing_date, "posted_text"].apply(get_date_from_text)

    india_time = df["posted_dt"].dt.tz_convert("Asia/Kolkata")
    df["posted_date_ist"] = india_time.dt.date
    df["posted_time_ist"] = india_time.dt.strftime("%H:%M:%S")

    today = pd.Timestamp.now(tz="Asia/Kolkata").normalize()
    df["days_since_posted"] = (
        today - india_time.dt.normalize()
    ).dt.days
    df = df.sort_values("posted_dt", ascending=False, na_position="last")
    df = df.reset_index(drop=True)
    df = df[[
        "title", "company", "location",
        "posted_date_ist", "posted_time_ist", "days_since_posted",
        "posted_text", "applicants", "jd", "link"
    ]]

In [10]:
file_name = f"linkedin_{job_title.replace(' ', '_')}_jobs.csv"
df.to_csv(file_name, index=False, encoding="utf-8-sig")

In [11]:
pd.set_option("display.max_colwidth", 150)
df[[
    "title", "company", "location", "posted_date_ist", "link"
]].head(10)

,title,company,location,posted_date_ist,link
0,"Right Shore, Revenue Planning Analyst",Teradata,"Navi Mumbai, Maharashtra, India",2026-09-26,https://in.linkedin.com/jobs/view/right-shore-revenue-planning-analyst-at-teradata-4470329603
1,"Index Projections and Analytics, BlackRock Global Markets (BGM), Associate",BlackRock,"Mumbai, Maharashtra, India",2026-09-26,https://in.linkedin.com/jobs/view/index-projections-and-analytics-blackrock-global-markets-bgm-associate-at-blackrock-4454531330
2,Climate Research Analyst,ISS STOXX,Mumbai Metropolitan Region,2026-09-26,https://in.linkedin.com/jobs/view/climate-research-analyst-at-iss-stoxx-4454524989
3,Index Data Research Analyst,ISS STOXX,Mumbai Metropolitan Region,2026-09-26,https://in.linkedin.com/jobs/view/index-data-research-analyst-at-iss-stoxx-4472016598
4,Data Management Executive – Backend,Altraize,Mumbai Metropolitan Region,2026-09-25,https://in.linkedin.com/jobs/view/data-management-executive-%E2%80%93-backend-at-altraize-4471006617
5,Business Insights Analyst,TaskUs,Mumbai Metropolitan Region,2026-09-25,https://in.linkedin.com/jobs/view/business-insights-analyst-at-taskus-4469798476
6,Analyst,Morningstar,"Mumbai, Maharashtra, India",2026-09-25,https://in.linkedin.com/jobs/view/analyst-at-morningstar-4467981067
7,Quant Analytics Associate -Digital,JPMorganChase,"Mumbai, Maharashtra, India",2026-09-25,https://in.linkedin.com/jobs/view/quant-analytics-associate-digital-at-jpmorganchase-4471173821
8,Business Analytics & Automation Associate (Onsite- Mumbai),Uplers,Mumbai Metropolitan Region,2026-09-25,https://in.linkedin.com/jobs/view/business-analytics-automation-associate-onsite-mumbai-at-uplers-4469835808
9,Supply Chain Analyst,Schreiber Foods,Mumbai Metropolitan Region,2026-09-25,https://in.linkedin.com/jobs/view/supply-chain-analyst-at-schreiber-foods-4442480306


In [12]:
driver.quit()
print("Browser closed.")


Browser closed.
